## Overvew




### Refs
- [video src](https://www.youtube.com/watch?v=kCc8FmEb1nY)

## Data Prep
### Steps
1. Download data
2. Tokenization
3. Torchify data
4. Split data into train and validation
    - Train: 90%
    - Validation: 10%
5. block_size

In [4]:
# ! means `pass the rest of the line as a system command`
## -nc --no-clobber (check the file existance)
## -P path to folder
!wget -nc -P ./assets https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

File ‘./assets/input.txt’ already there; not retrieving.



In [5]:
with open('./assets/input.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

In [10]:
text[:100], len(text)

('First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou',
 1115394)

### Tokenization
The tokenization is a process to transform string into a sequence of integer
- Overview
    - character level
    - If the vocab/dictionary size is larger then the length of token will be shorter (inversely proportional)
        -  dictionary size * lenght of token = C
- Tokenizer
    - [google sentence piece](https://github.com/google/sentencepiece)
    - [GPT tiktoken](https://github.com/openai/tiktoken?tab=readme-ov-file)


In [117]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"{'All Chrs:':<18}{''.join(chars)}")
print(f"{'Dictionary size:':<18}{vocab_size}")

All Chrs:         
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Dictionary size:  65


In [139]:
stoi = {char:i for i, char in enumerate(chars)}
itos = {i:char for i, char in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"{"Encode Hello World!:":<22}{encode('Hello World!')}")
print(f"{"Decode sequence:":<22}{decode(encode('Hello World!'))}")

Encode Hello World!:  [20, 43, 50, 50, 53, 1, 35, 53, 56, 50, 42, 2]
Decode sequence:      Hello World!


In [65]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
data.type, data[:30]

(<function Tensor.type>,
 tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
         53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43]))

In [66]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

## Block Size / Context Length
We set the block_size/context_size to 8. The model will be trained with the setting block_size/context_size = 8. Each training sequence contains 8 examples. So the sequcne is [c0, c1, c2, c3, c4, c5, c6, c7, c8] . 
The examples are
- [c0] -> [c1]
- [c0, c1] -> [c2]
- ...
- [c0, c1, c2, c3, c4, c5, c6, c7, c8] -> [c9]

In [72]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [110]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"context: {context}, target: {y[t]}")

context: tensor([18]), target: 47
context: tensor([18, 47]), target: 56
context: tensor([18, 47, 56]), target: 57
context: tensor([18, 47, 56, 57]), target: 58
context: tensor([18, 47, 56, 57, 58]), target: 1
context: tensor([18, 47, 56, 57, 58,  1]), target: 15
context: tensor([18, 47, 56, 57, 58,  1, 15]), target: 47
context: tensor([18, 47, 56, 57, 58,  1, 15, 47]), target: 58


### Batch
Why
- Utilize parallel programming of CPU
- Multiple chunks of texts are processed in parallel for single neuron  (batch_size chunk)

Spec
- batch_size = 4
- context_size = 8
- Each batch has 32 examples

In [98]:
batch_size = 4

def get_batch(split_on):
    data = train_data if split_on == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # So we left the last elemnt as target of last example
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')

# Shapes of xb and yb both are (4, 8) which is (B, T). B is batch index, T is the t-timestep of the sequence
xb, yb

(tensor([[39, 50, 49,  1, 53, 44,  1, 45],
         [41, 53, 61, 39, 56, 42, 50, 63],
         [43, 56, 58, 39, 47, 52,  1, 42],
         [43, 57, 57,  7,  7, 53, 40, 57]]),
 tensor([[50, 49,  1, 53, 44,  1, 45, 56],
         [53, 61, 39, 56, 42, 50, 63,  8],
         [56, 58, 39, 47, 52,  1, 42, 43],
         [57, 57,  7,  7, 53, 40, 57, 58]]))

In [123]:
xb.shape

torch.Size([4, 8])

In [111]:
for b in range(batch_size):
    x = xb[b]
    y = yb[b]
    for t in range(block_size):
        context = x[:t + 1]
        target = y[t]
        print(f"context: {context}, target: {y[t]}")

context: tensor([39]), target: 50
context: tensor([39, 50]), target: 49
context: tensor([39, 50, 49]), target: 1
context: tensor([39, 50, 49,  1]), target: 53
context: tensor([39, 50, 49,  1, 53]), target: 44
context: tensor([39, 50, 49,  1, 53, 44]), target: 1
context: tensor([39, 50, 49,  1, 53, 44,  1]), target: 45
context: tensor([39, 50, 49,  1, 53, 44,  1, 45]), target: 56
context: tensor([41]), target: 53
context: tensor([41, 53]), target: 61
context: tensor([41, 53, 61]), target: 39
context: tensor([41, 53, 61, 39]), target: 56
context: tensor([41, 53, 61, 39, 56]), target: 42
context: tensor([41, 53, 61, 39, 56, 42]), target: 50
context: tensor([41, 53, 61, 39, 56, 42, 50]), target: 63
context: tensor([41, 53, 61, 39, 56, 42, 50, 63]), target: 8
context: tensor([43]), target: 56
context: tensor([43, 56]), target: 58
context: tensor([43, 56, 58]), target: 39
context: tensor([43, 56, 58, 39]), target: 47
context: tensor([43, 56, 58, 39, 47]), target: 52
context: tensor([43, 56, 

In [140]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    # vocab_size is size of the all available chars
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):

        # idx and targets are both (B,T) tensor of integers
        # logits: (B,T,C) (Batch, Timestep, Channel = vocab_size) The case is (4, 8, 65) 
        logits = self.token_embedding_table(idx)

        if targets == None:
            loss = None
        else:
            # https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html
            # We can see that cross_entropy accept the input with dimension (B, C), so we need to rehape our logits and targets both to be (B, C)
            B, T, C = logits.shape
            logits = logits.view(B * T, C) # same as targets.view(-1, C)
            targets = targets.view(B * T) # same as targets.view(-1)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, _ = self(idx) # (B, T, C)
            # focus only on the last time step
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # -1 same as dim = 1 in the case, but -1 in pytorch means the channel dimension, since the channel is the last dimension
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples = 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
tensor(4.4743, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


- The initial loss should be -ln(1/65) = 4.16438...., so we have 4.4743 which means we have some sort of entropy.
- The `self(idx)` invoke `BigramLanguageModel.__call__` which is inherited from `nn.Module`. The method will invoke the prehook `forward`

- (TBU) # focus only on the last time step `logits = logits[:, -1, :] # becomes (B, C)`. It is just for simple illustration for bigram model. So for now we don't use any history of full context. But in then end, we will use. (How?)

In [141]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [146]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    # Reset all previous acuumulated gradients to 0
    optimizer.zero_grad(set_to_none=True)
    # Start backward propagation can calculate gradients for each computation node
    loss.backward()
    # Update params based on the gradients
    optimizer.step()

print(loss.item())

2.466055393218994


In [149]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype = torch.long), max_new_tokens = 500)[0].tolist()))


Anooun s, s, atrco thow
ARRUCKIs
JMedofuthe thed poouspelougad'd mur nd tit moumese brd ICKIININVIRES:

Tierte masomevengatter y y wigof thitr sticor t acof fr t kndrth OThthe abe woor-ther the mo ha ceer doulllitalowffsthendilis, jeefomenghaupicers hor, tshalleahe s'l,

O,

Ane f lis hmart happl fored ashofis VO fl mendo, tst s ilindl ENorputhize ndangl CKIDARYousegre gus, yo, fo we CENourkerst ove m my, the'shiedithil, myowind athe d' mamu. the vor ourchainow:
prthand
STorons d!
Thete megs bra


## Complete Code

In [151]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('./assets/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.7305, val loss 4.7241
step 300: train loss 2.8110, val loss 2.8249
step 600: train loss 2.5434, val loss 2.5682
step 900: train loss 2.4932, val loss 2.5088
step 1200: train loss 2.4863, val loss 2.5035
step 1500: train loss 2.4665, val loss 2.4921
step 1800: train loss 2.4683, val loss 2.4936
step 2100: train loss 2.4696, val loss 2.4846
step 2400: train loss 2.4638, val loss 2.4879
step 2700: train loss 2.4738, val loss 2.4911

MARI he avayokis erceller thour d, myono thishe me tord se by he me, Forder anen: at trselorinjulour t yoru thrd wo ththathy IUShe bavidelanoby man ond be jus as g e atot Meste hrle s, ppat t JLENCOLIUS:
Oppid tes d s o ged moer y pevehear soue maramapay fo t: bueyo malalyo!
Duir.
Fl ke it I t l o'ddre d ondu s?
cr, havetrathackes w.
PUpee meshancun, hrendspouthoulouren whel's'sesoread pe, s whure our heredinsethes; sedsend r lo pamit,
QUMIVIVIOfe m ne RDINid we tr ort; t:
MINENXI l dintandore r


## Side Notes
- Specify device so that your hardware will move the computation to GPU if it is available.
- estimate_loss is a way to calculate average loss, so it can illustrate the trend of overall loss.
- model.eval() vs. model.train()
    - trian
        - Batch Normalization (BatchNorm) layers use the statistics (mean and variance) of the current mini-batch to normalize inputs and update their running estimates for use during evaluation.
    - eval
        - Batch Normalization (BatchNorm) layers stop updating their running statistics and instead use the running mean and variance that were accumulated during the training phase. 